# Before / After TGLC — inspect the reprocessed light curves

For each reobserved target we take the **same** reprocessed FITS (which carries all cadences
through S103) and split it at the **S94 cadence boundary** to get:

- **before** = only sectors ≤ S93 (QLP-only, i.e. drop the new TGLC data)
- **after**  = everything (QLP ≤ S93 + TGLC ≥ S94)

Then we run the **real Astronet vetting preprocessing** (`detrend → fold → global/local view`)
on both and compare. This shows what actually changed by adding the TGLC sectors.

**Env:** run with your astronet dev env (e.g. conda `daniel_env_cloned_v2`). No database needed —
it only reads the FITS files in `reprocessed_s103_qlptglc_fits_files/` and imports
`astronet.preprocess` from `/pdo/users/pablomer/Astronet-Triage`.

Shared logic lives in `before_after_core.py` (same code the headless `diagnose_before_after.py` uses).

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, '/pdo/users/pablomer/TGLC_adaptation')
import importlib, before_after_core as core
importlib.reload(core)
import matplotlib.pyplot as plt

print('FITS dir:', core.FITS_DIR)
print('S94 cadence boundary:', core.S94_CADENCE_START, '(BTJD ~3856.26)')
comp, summ = core.load_tables()
print('companion TCEs:', len(comp), '| summary rows:', None if summ is None else len(summ))

In [ ]:
# Pick ~10 examples per class (finite ephemeris, sensible period, most TGLC-era cadences)
N_PER_CLASS = 10
examples = core.pick_examples(comp, summ, n_per_class=N_PER_CLASS)
for L, rows in examples.items():
    print(f"{core.CLASS_NAMES[L]:7s}: " + ', '.join(str(int(r['TIC ID'])) for r in rows))

In [ ]:
# Quick look at a single target (edit the TIC to inspect any one)
tic = int(examples['p'][0]['TIC ID'])
meta = comp[comp['TIC ID'] == tic].iloc[0]
fig, info = core.make_example_figure(tic, meta)
print(info)
plt.show()

In [ ]:
def show_class(L, rows=None):
    rows = rows or examples[L]
    print(f"===== {core.CLASS_NAMES[L]} ({len(rows)} examples) =====")
    for r in rows:
        tic = int(r['TIC ID'])
        try:
            fig, info = core.make_example_figure(tic, r)
            if info['errors']:
                print(f"  TIC {tic}: view errors -> {info['errors']}")
            plt.show()
        except Exception as e:
            print(f"  TIC {tic}: FAILED -> {e!r}")

## Planets

In [ ]:
show_class('p')

## Eclipsing binaries

In [ ]:
show_class('e')

## Junk

In [ ]:
show_class('j')